# Tugas Kelompok Sistem Rekomendasi
**Topik:** Prediksi Rating dan Rekomendasi Item
**Metode:** User-Based Collaborative Filtering, Item-Based Collaborative Filtering, Matrix Factorization (SGD)
**Dataset:** MovieLens (ml-latest-small) - Memiliki riwayat penilaian (user history)
**Metrik Evaluasi:** MAE, MSE, RMSE, NMSE, NDCG@10
**Referensi:** Buku Sistem Rekomendasi (Baizal & Agung Toto)

---
### Informasi Kelompok:
- **Nomor Kelompok:** [Isi Nomor Kelompok]
- **Anggota:**
  1. [Nama Anggota 1] - [NIM 1]
  2. [Nama Anggota 2] - [NIM 2]
  3. [Nama Anggota 3] - [NIM 3]
  4. [Nama Anggota 4] - [NIM 4]
  5. [Nama Anggota 5] - [NIM 5]
- **Akses Source Code:** Diberikan kepada bayzal@gmail.com

In [ ]:
import time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

## 1. Load Dataset MovieLens
Dataset memiliki riwayat penilaian lengkap: userId, movieId, rating, timestamp.

In [ ]:
ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv("movies.csv")
print(f"Jumlah rating : {len(ratings):,}")
print(f"Jumlah user   : {ratings.userId.nunique():,}")
print(f"Jumlah film   : {ratings.movieId.nunique():,}")
ratings.head()

## 2. Eksperimen & Evaluasi Tiga Metode
Modul `recsys.py` mengimplementasikan User-Based CF, Item-Based CF, dan Matrix Factorization (Funk-SVD SGD) secara modular dari scratch.

In [ ]:
from recsys import load_data, split_per_user, to_matrix, fit_baseline, baseline_matrix, user_similarity, user_cf_predict, item_similarity, item_cf_predict, mf_sgd, mf_full_matrix, evaluate_all, recommend

# Memuat data dan memetakan indeks
ratings, movies, uid, iid = load_data()
inv_iid = {v: k for k, v in iid.items()}
n_users, n_items = len(uid), len(iid)

# Hold-out per pengguna: 80% train, 20% test
train, test = split_per_user(ratings, 0.20, seed=SEED)
R_train = to_matrix(train, n_users, n_items)
print(f"Train size: {len(train):,} | Test size: {len(test):,}")

## 3. Hasil Pengujian dengan 5 Metrik Evaluasi
Metrik: MAE, MSE, RMSE, NMSE, dan NDCG@10.

In [ ]:
with open("results.json", "r") as f:
    res = json.load(f)

df_results = pd.DataFrame(res["final"]).T
df_results[["MAE", "MSE", "RMSE", "NMSE", "NDCG@10"]]

## 4. Contoh Rekomendasi Top-10
Daftar rekomendasi 10 item terbaik untuk pengguna contoh.

In [ ]:
demo_u = res["demo_user"]
print(f"=== Top 10 Rekomendasi Film untuk User ID: {demo_u} ===\n")
recs = res["recs"]
top10_df = pd.DataFrame({
    "User-based CF": [f"{r['title']} ({r['pred']:.2f})" for r in recs["User-based CF"]],
    "Item-based CF": [f"{r['title']} ({r['pred']:.2f})" for r in recs["Item-based CF"]],
    "Matrix Factorization": [f"{r['title']} ({r['pred']:.2f})" for r in recs["Matrix Factorization"]],
}, index=range(1, 11))
top10_df